# Explorer Tutorial

This tutorial demonstrates how to use the Explorer components of the EpiScope package.  Explorer implements retrieval‑augmented generation (RAG) over indexed documents.  In this example we focus on computing simple embeddings and augmenting a query using a hypothetical document (HYDE) generator.  For brevity, we do not perform actual indexing into a vector store.

## Compute Embeddings with SimplifiedEmbedder

The `SimplifiedEmbedder` class wraps a HuggingFace embedding model and provides batched encoding.  When the underlying embedding library (llama_index/transformers) is unavailable, the embedder falls back to a stub implementation that returns a vector based on the length of the input text.  This makes the embedder useful even in a test environment.  Below we instantiate an embedder and compute embeddings for a list of sentences.

In [1]:
from episcope.retrieve.embeddings import SimplifiedEmbedder

# Instantiate a simplified embedder (falls back to a stub if transformers# is unavailable).  The `dim` attribute reflects the hidden size of the# underlying model or 1 for the stub.
embedder = SimplifiedEmbedder(embed_model='distilbert-base-uncased', batch_size=2)
print(f'Embedding dimension: {embedder.dim}')

sentences = [
    'Influenza spreads quickly in winter.',
    'Vaccination reduces the risk of infection.',
    'Personal protective equipment (PPE) helps prevent transmission.'
]
vectors = embedder.embed_texts(sentences)
for s, v in zip(sentences, vectors):
    print(f'Sentence: {s} Embedding: {v}')


ModuleNotFoundError: No module named 'episcope'

## Enhance a Query with HYDE

The HYDE (Hypothetical Document) generator augments queries by generating a hypothetical document describing what an answer might look like.  This can improve retrieval performance by providing richer context.  In this example we patch the underlying LLM call to return a deterministic response for illustration.

In [ ]:
from unittest.mock import patch
from episcope.core.hyde import HYDE

# Patch ollama.chat so that HYDE.generate returns a predictable string
def fake_chat(model_name: str, messages: list, options: dict):
    return {'message': {'content': 'This is a hypothetical document about the query.'}}

with patch('episcope.core.hyde.ollama.chat', side_effect=fake_chat):
    hyde = HYDE(model_name='tinyllama')
    query = 'What is the basic reproduction number (R0) of a virus?'
    print('Original query:', query)
    hypothetical = hyde.generate(query)
    print('Generated hypothetical document:', hypothetical)
    enhanced = hyde.enhance_query(query, paper_title=None, domain=None)
    print('Enhanced query with HYDE:', enhanced)

## Next Steps

In a full Explorer pipeline you would index your PDF documents (using `TextRAG.index`) and then perform retrieval with `TextRAG.retrieve`.  The retrieved contexts can then be passed to an LLM to generate answers with provenance.  This notebook introduced the basic building blocks (embedding and query augmentation) to get you started.

## Loading Documents for Indexing

Before you can index documents for retrieval you must first extract their text content.  EpiScope includes a document loader factory that supports multiple backends (Unstructured, GROBID).  In this example we show how to load a plain text file into a list of structured sections.  These sections can then be passed to the `PaperIndexer` for per‑paper indexing.

In [1]:
from episcope.ingest import DocumentLoaderFactory
from episcope.index.paper_indexer import PaperIndexer

# Create a loader (Unstructured is used by default for PDFs and text).
loader = DocumentLoaderFactory.get_loader('unstructured')
# Assume we have a simple text file on disk.  For this demonstration
# we create a temporary file, but in practice you would provide a
# path to a .pdf or .txt document.
import tempfile
temp = tempfile.NamedTemporaryFile(delete=False, suffix='.txt')
temp.write(b'First paragraph.\n\nSecond paragraph.')
temp.close()

sections, metadata = loader.load(temp.name)
print('Loaded', len(sections), 'section(s) from', temp.name)
for sec in sections:
    print('-', sec.title or '<no title>')
    print(sec.content)

# Index the sections using a PaperIndexer
indexer = PaperIndexer(min_chunk_size=5)
indexer.index_paper(sections, metadata, paper_id='TEMP_DOC')
# You can now search this paper using indexer.search()
print('Top result:', indexer.search('First paragraph')[0])


ModuleNotFoundError: No module named 'episcope'